## our imports


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, RandomFlip, RandomRotation, Lambda
from tensorflow.keras.models import Model
# this is our EfficientNet Imports
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess_input
# we just importing the standard
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import cv2 # here we are  importinghte opencv

## Connect to Google Drive


In [ ]:
#  hew we are  Connecting to Google Drive
print("Connecting to Google Drive...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted successfully.")
except ImportError:
    print("Google Colab not detected. Skipping drive mount.")

## Unzip our dataset

In [ ]:
# here we are Unziping  our dataset
print("Unzipping the DATA.zip file from Drive...")
ZIP_PATH = "/content/drive/MyDrive/DATA.zip"
# Unzip quietly (-q) and overwrite (-o)
!unzip -o -q {ZIP_PATH} -d "/content/"
print("Data is unzipped and ready in /content/.")

## Define our project variables


In [ ]:
# here we are defining  the variable
IMAGE_SIZE = (224, 224) # here is our default input size for the EfficientNetB0
BATCH_SIZE = 32
NUM_BINARY_CLASSES = 1 #just simple  output neuron for sigmoid activation
EPOCHS_STAGE_1 = 15
EPOCHS_STAGE_2 = 10

# hree we are just  Defining  the paths to our unzipped data folders
TRAIN_DIR = "/content/DATA/Training(70%)"
VALID_DIR = "/content/DATA/Validation(20%)"
TEST_DIR = "/content/DATA/Testing(10%)"

## Define Preprocessing Functions (Part 1: OpenCV)

In [ ]:
# here we are defining preprocessing functions

# here we are creating a function that runs in numpy
def apply_preprocessing(image_array):
    # here we are converting the image from float32 to uint8 for opencv
    image_uint8 = image_array.astype(np.uint8)

    # here we are removing noise using median blur
    image_blur = cv2.medianBlur(image_uint8, 5)

    # here we are enhancing contrast using clahe on the l channel
    image_lab = cv2.cvtColor(image_blur, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(image_lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel)

    # here we are merging the channels back and converting to rgb
    merged = cv2.merge((cl, a_channel, b_channel))
    final_image = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

    # here we are returning the processed image as float32
    return final_image.astype(np.float32)


## Preprocessing (Part 2: TensorFlow Wrapper)

In [ ]:
# here we are defining a wrapper that uses the correct tf numpy function
@tf.function
def tf_preprocess_wrapper(image, label):
    # here we are using tf numpy function to apply our opencv preprocessing
    [image,] = tf.numpy_function(
        apply_preprocessing,
        [image],
        [tf.float32]
    )

    # here we are setting the image shape back to the correct size
    image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    return image, label


## Custom Data Pipeline for Binary Labels

In [ ]:
# here we are defining a custom data pipeline for binary labels
@tf.function
def map_to_binary_label(image, multi_class_label):
    # here we are getting the value of no tumor from index 2
    no_tumor_value = multi_class_label[2]

    # here we are creating the binary label where 1 means tumor and 0 means no tumor
    binary_label = 1.0 - no_tumor_value

    # here we are making sure the binary label is a float32 tensor with shape 1
    binary_label = tf.expand_dims(binary_label, axis=0)
    return image, binary_label


## Create Data Pipelines (Load Data)


In [ ]:
# here we are creating data pipelines with preprocessing
print("loading training, validation, and test datasets...")

# here we are loading the training dataset with categorical labels
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, label_mode="categorical", seed=123,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    interpolation='bilinear'
)

# here we are loading the validation dataset
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR, label_mode="categorical", seed=123,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    interpolation='bilinear'
)

# here we are loading the test dataset without shuffling
test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, label_mode="categorical", image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=False, interpolation='bilinear'
)

# here we are getting the class names and checking their order
class_names = train_dataset.class_names
print(f"original class names found: {class_names}")
print(f"confirming no_tumor is at index 2: {class_names[2] == 'no_tumor'}")


## Apply All Preprocessing to Pipelines

In [ ]:
# here we are optimizing data pipelines and applying preprocessing
print("optimizing data pipelines and applying preprocessing...")

AUTOTUNE = tf.data.AUTOTUNE

# here we are defining a function to preprocess and map binary labels
def preprocess_and_map_binary(ds):
    ds = ds.unbatch()
    ds = ds.map(tf_preprocess_wrapper, num_parallel_calls=AUTOTUNE)
    ds = ds.map(map_to_binary_label, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).cache().prefetch(buffer_size=AUTOTUNE)
    return ds

# here we are applying preprocessing to train, validation, and test datasets
train_dataset_binary = preprocess_and_map_binary(train_dataset)
validation_dataset_binary = preprocess_and_map_binary(validation_dataset)
test_dataset_binary = preprocess_and_map_binary(test_dataset)

# here we are confirming that preprocessing and binary mapping are applied to all datasets
print("preprocessing and binary mapping functions have been mapped to all datasets")


## Build and Train (Part 1: Build the Model)

In [ ]:
# here we are building and training the binary efficientnetb0 model
print("building the binary efficientnetb0 model...")

# here we are loading the efficientnetb0 base model with imagenet weights
base_model_effnet = EfficientNetB0(weights='imagenet', include_top=False,
                                   input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

# here we are freezing the base model for stage 1 training
base_model_effnet.trainable = False

# here we are preparing the input and applying data augmentation
inputs = Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
x = RandomFlip('horizontal')(inputs)
x = RandomRotation(0.1)(x)

# here we are preprocessing the input using the efficientnet function
x = Lambda(effnet_preprocess_input)(x)

# here we are passing the data through the base model
x = base_model_effnet(x, training=False)

# here we are adding global average pooling and dropout
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)

# here we are adding the binary classifier head
outputs = Dense(NUM_BINARY_CLASSES, activation='sigmoid')(x)

# here we are creating the final binary model
model_effnet_binary = Model(inputs, outputs)


## Train Stage 1 (Head Only)

In [ ]:
# here we are compiling the model for stage 1
model_effnet_binary.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# here we are starting the training for stage 1
print("starting efficientnet binary model training stage 1 head only")
history_effnet = model_effnet_binary.fit(
    train_dataset_binary,
    validation_data=validation_dataset_binary,
    epochs=EPOCHS_STAGE_1,
    verbose=1
)

# here we are finishing the training for stage 1
print("efficientnet binary stage 1 training complete")


## Stage 2: Fine-Tuning Setup

In [ ]:
# here we are starting the fine tuning stage for the efficientnet model
print("unfreezing the top 30 layers of the efficientnet model...")

# making the base model trainable again
base_model_effnet.trainable = True

# keeping all layers except the last 30 frozen so that training focuses on top layers
for layer in base_model_effnet.layers[:-30]:
    layer.trainable = False

# now we compile the model again for fine tuning using a small learning rate
model_effnet_binary.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

# confirming that the model has been recompiled for fine tuning
print("model recompiled for fine tuning")

# showing the updated summary of the model after changes
model_effnet_binary.summary()


## Train Stage 2 (Fine-Tuning)

In [ ]:
print("Continuing training to fine-tune the unfrozen layers ")
history_finetune_effnet = model_effnet_binary.fit(
    train_dataset_binary,
    validation_data=validation_dataset_binary,
    epochs=EPOCHS_STAGE_2,
    initial_epoch=history_effnet.epoch[-1], # here we are using new story variable
    verbose=1
)
print(" EfficientNet Binary Stage 2 Fine-Tuning Complete ")

## . Final Evaluation


In [ ]:
# we are doinght the Final Evaluation and Saving
print("\n--- Evaluating the EfficientNet Binary Model on the Test Set ---")
results_effnet = model_effnet_binary.evaluate(test_dataset_binary, verbose=1)

# here we are Calculating  F1-Score
metrics_effnet = {
    'loss': results_effnet[0],
    'accuracy': results_effnet[1],
    'precision': results_effnet[2],
    'recall': results_effnet[3]
}
if (metrics_effnet['precision'] + metrics_effnet['recall']) > 0:
    metrics_effnet['f1_score'] = 2 * (metrics_effnet['precision'] * metrics_effnet['recall']) / (metrics_effnet['precision'] + metrics_effnet['recall'])
else:
    metrics_effnet['f1_score'] = 0.0

print("\n--- EFFICIENTNET BINARY Fine-Tuned Model Test Results ---")
print(f"Test Loss: {metrics_effnet['loss']:.4f}")
print(f"Test Accuracy: {metrics_effnet['accuracy']:.4f}")
print(f"Test Precision: {metrics_effnet['precision']:.4f}")
print(f"Test Recall: {metrics_effnet['recall']:.4f}")
print(f"Test F1-Score: {metrics_effnet['f1_score']:.4f}")

## Save the Final Model


In [ ]:
#  Save our final binary model
print("\n Saving the EfficientNet binary model to Google Drive ")
os.makedirs("/content/drive/MyDrive/MODELS", exist_ok=True)
model_effnet_binary.save("/content/drive/MyDrive/MODELS/efficientnet_binary_finetuned.h5")

print("EfficientNet Binary Fine-Tuned model saved. Experiment Complete.")

## might delete later


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, RandomFlip, RandomRotation, Lambda
from tensorflow.keras.models import Model
# --- EfficientNet Imports ---
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess_input
# --- Standard Imports ---
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import cv2 # For OpenCV

# --- 1. Connect to Google Drive ---
print("Connecting to Google Drive...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted successfully.")
except ImportError:
    print("Google Colab not detected. Skipping drive mount.")

# --- 2. Unzip our dataset ---
print("Unzipping the DATA.zip file from Drive...")
ZIP_PATH = "/content/drive/MyDrive/DATA.zip"
# Unzip quietly (-q) and overwrite (-o)
!unzip -o -q {ZIP_PATH} -d "/content/"
print("Data is unzipped and ready in /content/.")

# --- 3. Define our project variables ---
IMAGE_SIZE = (224, 224) # EfficientNetB0 default input size
BATCH_SIZE = 32
NUM_BINARY_CLASSES = 1 # 1 output neuron for sigmoid activation
EPOCHS_STAGE_1 = 15
EPOCHS_STAGE_2 = 10

# --- Define the paths to our unzipped data folders ---
TRAIN_DIR = "/content/DATA/Training(70%)"
VALID_DIR = "/content/DATA/Validation(20%)"
TEST_DIR = "/content/DATA/Testing(10%)"

# --- 4. Define Preprocessing Functions ---

# This function will run in NumPy
def apply_preprocessing(image_array):
    # image_array comes in as float32 (0-255)
    # We MUST convert to uint8 for OpenCV operations
    image_uint8 = image_array.astype(np.uint8)

    # 1. Noise Removal (now on a uint8 image)
    image_blur = cv2.medianBlur(image_uint8, 5)

    # 2. Contrast Enhancement (now on a uint8 image)
    image_lab = cv2.cvtColor(image_blur, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(image_lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel) # This will now work (uint8)

    merged = cv2.merge((cl, a_channel, b_channel))
    final_image = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

    # Return as float32 for the model
    return final_image.astype(np.float32)

# This wrapper uses the correct tf.numpy_function
@tf.function
def tf_preprocess_wrapper(image, label):
    # We use tf.numpy_function to run our OpenCV code
    [image,] = tf.numpy_function(
        apply_preprocessing, # Our fixed function
        [image],             # Input tensor (float32, 0-255)
        [tf.float32]         # Output data type (float32)
    )
    # We must manually set the shape back
    image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    return image, label

# --- 5. Custom Data Pipeline for Binary Labels ---
@tf.function
def map_to_binary_label(image, multi_class_label):
    """
    Derives the binary label (0 or 1) from the 4-class one-hot tensor.
    Based on prior analysis, 'no_tumor' is at index 2.
    """
    # Get the 'No Tumor' value (index 2)
    no_tumor_value = multi_class_label[2] # <-- CRITICAL INDEX FIX

    # Binary Label = 1 (Tumor) - No Tumor Value
    binary_label = 1.0 - no_tumor_value

    # Ensure the binary label is a float32 tensor of shape (1,)
    binary_label = tf.expand_dims(binary_label, axis=0)
    return image, binary_label

# --- 6. Create Data Pipelines (with Preprocessing) ---
print("Loading Training, Validation, and Test datasets...")
# We load the data as-is (float32, 0-255) and label_mode='categorical'
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, label_mode="categorical", seed=123,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    interpolation='bilinear'
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR, label_mode="categorical", seed=123,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    interpolation='bilinear'
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, label_mode="categorical", image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=False, interpolation='bilinear'
)

class_names = train_dataset.class_names
print(f"Original class names found: {class_names}")
print(f"Confirming 'no_tumor' is at index 2: {class_names[2] == 'no_tumor'}")

# --- Optimize & Apply Preprocessing and Binary Mapping ---
print("Optimizing data pipelines and applying preprocessing...")
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_and_map_binary(ds):
    ds = ds.unbatch()
    ds = ds.map(tf_preprocess_wrapper, num_parallel_calls=AUTOTUNE) # Apply CV2 Preprocessing
    ds = ds.map(map_to_binary_label, num_parallel_calls=AUTOTUNE)   # Map to Binary Label
    ds = ds.batch(BATCH_SIZE).cache().prefetch(buffer_size=AUTOTUNE)
    return ds

train_dataset_binary = preprocess_and_map_binary(train_dataset)
validation_dataset_binary = preprocess_and_map_binary(validation_dataset)
test_dataset_binary = preprocess_and_map_binary(test_dataset)

print("Preprocessing and binary mapping functions have been mapped to all datasets.")

# --- 7. Build and Train Binary Model (EfficientNetB0) ---
print("\nBuilding the Binary EfficientNetB0 model...")
# Load the EfficientNetB0 base model
base_model_effnet = EfficientNetB0(weights='imagenet', include_top=False,
                                 input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

# --- Stage 1: Head Training (Frozen Base) ---
base_model_effnet.trainable = False
inputs = Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
x = RandomFlip('horizontal')(inputs)
x = RandomRotation(0.1)(x)

# --- KEY CHANGE: Use the specific EfficientNet preprocess_input function ---
x = Lambda(effnet_preprocess_input)(x)

x = base_model_effnet(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)

# Binary Classifier Head (1 neuron, sigmoid)
outputs = Dense(NUM_BINARY_CLASSES, activation='sigmoid')(x)
model_effnet_binary = Model(inputs, outputs)

# Compile for Stage 1 (Binary Loss)
model_effnet_binary.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\n--- Starting EfficientNet Binary Model Training (Stage 1: Head Only) ---")
history_effnet = model_effnet_binary.fit(
    train_dataset_binary,
    validation_data=validation_dataset_binary,
    epochs=EPOCHS_STAGE_1,
    verbose=1
)
print("--- EfficientNet Binary Stage 1 Training Complete ---")

# --- 8. Stage 2: Fine-Tuning ---
print("Unfreezing the top 30 layers of the EfficientNet model...")
base_model_effnet.trainable = True
# We use the same 30-layer unfreeze strategy for a fair comparison
for layer in base_model_effnet.layers[:-30]:
    layer.trainable = False

# Re-compile for fine-tuning
model_effnet_binary.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

print("--- Model Re-compiled for Fine-Tuning ---")
model_effnet_binary.summary()

print("Continuing training to fine-tune the unfrozen layers...")
history_finetune_effnet = model_effnet_binary.fit(
    train_dataset_binary,
    validation_data=validation_dataset_binary,
    epochs=EPOCHS_STAGE_2,
    initial_epoch=history_effnet.epoch[-1], # Use the new history variable
    verbose=1
)
print("--- EfficientNet Binary Stage 2 Fine-Tuning Complete ---")

# --- 9. Final Evaluation and Saving ---
print("\n--- Evaluating the EfficientNet Binary Model on the Test Set ---")
results_effnet = model_effnet_binary.evaluate(test_dataset_binary, verbose=1)

# Calculate F1-Score
metrics_effnet = {
    'loss': results_effnet[0],
    'accuracy': results_effnet[1],
    'precision': results_effnet[2],
    'recall': results_effnet[3]
}
if (metrics_effnet['precision'] + metrics_effnet['recall']) > 0:
    metrics_effnet['f1_score'] = 2 * (metrics_effnet['precision'] * metrics_effnet['recall']) / (metrics_effnet['precision'] + metrics_effnet['recall'])
else:
    metrics_effnet['f1_score'] = 0.0

print("\n--- EFFICIENTNET BINARY Fine-Tuned Model Test Results ---")
print(f"Test Loss: {metrics_effnet['loss']:.4f}")
print(f"Test Accuracy: {metrics_effnet['accuracy']:.4f}")
print(f"Test Precision: {metrics_effnet['precision']:.4f}")
print(f"Test Recall: {metrics_effnet['recall']:.4f}")
print(f"Test F1-Score: {metrics_effnet['f1_score']:.4f}")

# --- Save our final binary model ---
print("\n--- Saving the EfficientNet binary model to Google Drive ---")
os.makedirs("/content/drive/MyDrive/MODELS", exist_ok=True)
model_effnet_binary.save("/content/drive/MyDrive/MODELS/efficientnet_binary_finetuned.h5")

print("EfficientNet Binary Fine-Tuned model saved. Experiment Complete.")

Connecting to Google Drive...
Mounted at /content/drive
Drive mounted successfully.
Unzipping the DATA.zip file from Drive...
Data is unzipped and ready in /content/.
Loading Training, Validation, and Test datasets...
Found 2297 files belonging to 4 classes.
Found 573 files belonging to 4 classes.
Found 394 files belonging to 4 classes.
Original class names found: ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
Confirming 'no_tumor' is at index 2: True
Optimizing data pipelines and applying preprocessing...
Preprocessing and binary mapping functions have been mapped to all datasets.

Building the Binary EfficientNetB0 model...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- Starting EfficientNet Binary Model Training (Stage 1: Head Only) ---
Epoch 1/15
     72/Unknown 25s 147ms/step - accuracy: 0.8430 - loss: 0.3857

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


72/72 ━━━━━━━━━━━━━━━━━━━━ 30s 227ms/step - accuracy: 0.8435 - loss: 0.3846 - val_accuracy: 0.9005 - val_loss: 0.2392
Epoch 2/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 8s 114ms/step - accuracy: 0.9130 - loss: 0.2187 - val_accuracy: 0.9459 - val_loss: 0.1849
Epoch 3/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 6s 83ms/step - accuracy: 0.9433 - loss: 0.1597 - val_accuracy: 0.9476 - val_loss: 0.1587
Epoch 4/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 6s 88ms/step - accuracy: 0.9442 - loss: 0.1392 - val_accuracy: 0.9494 - val_loss: 0.1448
Epoch 5/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 6s 83ms/step - accuracy: 0.9498 - loss: 0.1422 - val_accuracy: 0.9529 - val_loss: 0.1353
Epoch 6/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 6s 86ms/step - accuracy: 0.9548 - loss: 0.1256 - val_accuracy: 0.9529 - val_loss: 0.1268
Epoch 7/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 6s 83ms/step - accuracy: 0.9582 - loss: 0.1176 - val_accuracy: 0.9546 - val_loss: 0.1220
Epoch 8/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 6s 86ms/step - accuracy: 0.9625 - loss: 0.1083 - val_accuracy: 0.9546 - val_loss:

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,497,441 (5.71 MB)

 Non-trainable params: 2,553,411 (9.74 MB)

Continuing training to fine-tune the unfrozen layers...
--- EfficientNet Binary Stage 2 Fine-Tuning Complete ---

--- Evaluating the EfficientNet Binary Model on the Test Set ---
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 155ms/step - accuracy: 0.8403 - loss: 0.4034 - precision: 0.9565 - recall: 0.8505



--- EFFICIENTNET BINARY Fine-Tuned Model Test Results ---
Test Loss: 0.4063
Test Accuracy: 0.8401
Test Precision: 0.9154
Test Recall: 0.8616
Test F1-Score: 0.8877

--- Saving the EfficientNet binary model to Google Drive ---
EfficientNet Binary Fine-Tuned model saved. Experiment Complete.
